In [ ]:
from aereo.builtins.search import search_stac
from aereo.builtins.task_builder import build_grouped_tasks
from aereo.cache import TaskResultCache
from aereo.executors import LocalExecutor
from aereo.pipeline import ExtractionJob

job = ExtractionJob.load_from_config(
    config_dir="config",
    config_name="job_ndvi",
)

In [ ]:
from datetime import datetime, timezone

assets = search_stac(
    stac_api_url="https://earth-search.aws.element84.com/v1",
    collections={"sentinel-2-l2a": ["red", "nir"]},
    intersects="config/aoi/chocon.geojson",
    start_datetime=datetime(2024, 1, 1, tzinfo=timezone.utc),
    end_datetime=datetime(2024, 1, 10, 23, 59, 59, tzinfo=timezone.utc),
)
print(f"✓ Found {len(assets)} scenes")

In [ ]:
tasks = build_grouped_tasks(
    search_results=assets,
    job=job,
    cells_per_task=5,
)
print(f"✓ Built {len(tasks)} tasks")

In [ ]:
artifacts = job.execute(
    tasks,
    executor=LocalExecutor(workers=8, cache=TaskResultCache()),
)
print(f"✓ Extracted {len(artifacts)} artifacts")

In [ ]:
from aereo.viz import plot_artifact_patches

plot_artifact_patches(artifacts, cmap="RdBu_r", colorbar_label="ndvi")